Using the file CompiledCANONInfo.txt, use the OpenAI API to train a chatbot that can answer questions about Star Wars


In [1]:
import openai
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_openai import OpenAIEmbeddings
from langchain.chains.question_answering import load_qa_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import GPT4AllEmbeddings 
from langchain_community.llms import GPT4All
from langchain.chains import RetrievalQA

In [2]:
# Load the .env file
load_dotenv()
# Load the OpenAI API key from the .env file
openai.api_key = os.getenv("OPENAI_API_KEY")
# Load the .txt file using the UnstructuredTextLoader
print("Loading data...")
loader = UnstructuredFileLoader("CompiledALLInfo.txt")
data = loader.load()
print("Data loaded.")

Loading data...
Data loaded.


In [3]:
persist_directory = "vectDBALLINFO"
embedding = GPT4AllEmbeddings()#OpenAIEmbeddings() 
def generateVectorDB(data):
    print("Splitting document...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
    documents = text_splitter.split_documents(data)
    print("Number of documents: ", len(documents))
    print("Documents split.")
    """TODO: GET AN EMBEDDING THAT'LL LET ME USE THE VECTOR STORES"""
    print("Generating vector database...")
    db = Chroma.from_documents(documents, embedding, persist_directory=persist_directory)
    db.persist()
    print("Vector database created.")
generateVectorDB(data)
# print(docs[0].page_content)

Splitting document...
Number of documents:  950938
Documents split.
Generating vector database...
Vector database created.


In [4]:
# Now, we can use the QA chain to answer questions
# TODO: Setup database retrieval to then use to query and generate an answer
db = Chroma(persist_directory=persist_directory,embedding_function=embedding)
retriever = db.as_retriever(search_type="similarity")
print("Retriever created.")
myLLM = GPT4All(model="models/mistral-7b-openorca.Q4_0.gguf", n_threads=8)
print("LLM created.")
qa = RetrievalQA.from_chain_type(llm=myLLM, chain_type="map_reduce", retriever=retriever, return_source_documents=True)
query = "Who is Luke Skywalker?"
result = qa({"query": query})
print(result)

Retriever created.
LLM created.


c:\Users\zs811\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_core\_api\deprecation.py:117: LangChainDeprecationWarning: The function `__call__` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(
c:\Users\zs811\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Token indices sequence length is longer than the specified maximum sequence length for this model (1502 > 1024). Running this sequence through the model will result in indexing errors


{'query': 'Who is Luke Skywalker?', 'result': ' Luke Skywalker is a fictional character in the Star Wars universe created by George Lucas, portrayed by Mark Hamill in the original trilogy.', 'source_documents': [Document(page_content='Luke Skywalker. [1]', metadata={'source': 'CompiledALLInfo.txt'}), Document(page_content='Luke Skywalker. [1]', metadata={'source': 'CompiledALLInfo.txt'}), Document(page_content='who played Luke Skywalker in the original trilogy. [29]', metadata={'source': 'CompiledALLInfo.txt'}), Document(page_content='who played Luke Skywalker in the original trilogy. [29]', metadata={'source': 'CompiledALLInfo.txt'})]}
